In [1]:
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
import os
import io
import builtins
import time
import re
from itertools import product
import dash
from dash import dcc
from dash import html 
from dash import dash_table
import dash_bootstrap_components as dbc
from datetime import datetime


from dash.dependencies import Input, Output
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 63)

In [2]:
python_exports = "C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports"
current_save = "Havant"
all_players_file = "all_players_data.csv"

In [3]:
folder_path = python_exports + "/" + current_save
all_players_file_path = folder_path + "/" + all_players_file
all_folder_files = os.listdir(folder_path)


In [4]:
all_html_files = [f for f in all_folder_files if f.endswith('.html')]    
all_html_files = [w.replace('.html', '') + ' 00:00:01' for w in all_html_files]


In [5]:
if not os.path.exists(all_players_file_path):
    exported_dates = pd.DataFrame({'Date': [1, 2]})

else:
    # If the file exists, read the existing data and make a copy    
    exported_dates = pd.read_csv(all_players_file_path)

exported_dates = exported_dates.Date.unique().tolist()
print(exported_dates)

['2024-07-01 00:00:01', '2024-10-01 00:00:01', '2025-01-01 00:00:01', '2025-04-01 00:00:01', '2025-07-01 00:00:01', '2025-10-01 00:00:01', '2026-01-01 00:00:01', '2026-04-01 00:00:01', '2026-07-01 00:00:01', '2026-10-01 00:00:01', '2027-01-01 00:00:01', '2027-04-01 00:00:01', '2027-07-01 00:00:01', '2027-10-01 00:00:01', '2028-01-01 00:00:01', '2028-04-01 00:00:01', '2028-07-01 00:00:01', '2028-10-01 00:00:01', '2029-01-01 00:00:01', '2029-04-01 00:00:01', '2029-07-01 00:00:01', '2029-10-01 00:00:01']


In [6]:
for file in all_html_files:
    if file not in exported_dates:
        file_name = file.replace(' 00:00:01','')
        full_path = folder_path + '/' + file_name + '.html'
        fm_export= bs(open(full_path,encoding = "utf8"), "html.parser").find('table')
        raw_list = pd.read_html(io.StringIO(str(fm_export)))
        raw_table = pd.DataFrame(raw_list[0])
        date = file_name.split('.')[0] +  ' 00:00:01'
        raw_table['Date'] = pd.to_datetime(date, format = "%Y-%m-%d %H:%M:%S")
        raw_table['DoB'] = pd.to_datetime(raw_table["DoB"].str.split(" ").str[0], format="%d/%m/%Y")
        raw_table['DoB'] = raw_table['DoB'] + pd.Timedelta(seconds=1)
        raw_table['Age'] = (raw_table['Date']-raw_table['DoB']).dt.days/365.25
        raw_table['Acc'] = pd.to_numeric(raw_table['Acc'], errors='coerce')

        raw_table = raw_table.dropna(subset=['Acc'])
        if not os.path.exists(all_players_file_path):
            raw_table.to_csv(all_players_file_path, index=False)
            print(f"Created file: {all_players_file_path}")
        else:
            # If the file exists, read the existing data and make a copy
            timestr = time.strftime("%Y%m%d-%H%M%S")
            existing_df = pd.read_csv(all_players_file_path)
            csv_filename = f'C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/{current_save}/all_players_data_{timestr}.csv'
            existing_df.to_csv(csv_filename, index=False)

            # Union the existing DataFrame with the new DataFrame
            combined_df = pd.concat([existing_df, raw_table]).drop_duplicates().reset_index(drop=True)

            # Save the combined DataFrame back to the CSV file
            combined_df.to_csv(all_players_file_path, index=False)
            print(f"Updated file: {all_players_file_path}")


Updated file: C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/Havant/all_players_data.csv


In [7]:
full_path = folder_path + '/' + 'all_players_data' + '.csv'

In [8]:
raw_table = pd.read_csv(full_path)


In [9]:
def process_segment(segment):
    segment = segment.replace("D/","Z/").replace("D ",("Z "))
    sides = re.findall(r'\((.*?)\)', segment)
    sides = sides[0]    
    lines = segment.replace("("+sides+")","").replace("/","").replace("AM","A").replace("DM","O").replace('ST','F').replace("WB","O").replace("Z","D").strip()
    result = [f"{x}{y}" for x, y in product(lines, sides)] 
    return result

In [10]:
def combine_segment(segments):
    result = []
    segments = segments.replace('GK','G (C)')
    segments = re.sub(r'\bDM\b(,?)', r'DM (C)\1', segments)
    for segment in segments.split(','):
        result.extend(process_segment(segment.strip()))
    return result

In [11]:
def apply_combinations(df, column):

    df["Positions"] = df[column].apply(combine_segment)
    df['Positions'] = df['Positions'].apply(replace_values)

    return df


In [12]:
def replace_values(lst):
    new_lst = []
    for item in lst:
        item = item.replace('O', 'DM')
        item = item.replace('GC', 'GK')
        item = item.replace('FC', 'ST')
        item = item.replace('A', 'AM')
        item = item.replace('DMR', 'WBR')
        item = item.replace('DML', 'WBL')
        item = item.replace('DMC', 'DM')


        new_lst.append(item)
    return new_lst

In [20]:
df = apply_combinations(raw_table, 'Position')

In [21]:
latest_export_date = df['Date'].max()
df['gk_rating'] = ((5 * (df['Agi']+df['Ref']+ df['Cmd']+df['Cnt'])) + (3 * (df['1v1']+df['Ant']+df['Kic']+df['Pos'])) +(1 * (df['Acc']+df['Aer']+df['Dec']+df['Fir']+df['Han']+df['Pas']+df['TRO']+df['Vis'])))/40
df['of_rating'] = (((df['Acc']+ df['Pac'])*5) +  ((df['Dri']+ df['Ant'] + df['Cnt']+ df['Agi']+df['Bal']+ df['Jum'])*3) + (df['Det']+ df['Wor'] + df['Sta']+ df['Nat.1']))/32
df['rating'] = df['gk_rating'].where(df['Positions'] == 'GK', df['of_rating'])
df['physical'] = df['Acc']+df['Pac'] + df['Jum']
earliest_ratings = df.loc[df.groupby('UID')['Date'].idxmin(), ['UID', 'rating','physical']].rename(columns={'rating': 'first_rating','physical': 'first_physical'})
df = df.merge(earliest_ratings, on='UID', how='left')

df['rating_growth'] = df['rating'] - df['first_rating']
df['physical_growth'] = df['physical'] - df['first_physical']

df['age_at_export'] = df['Age'].apply(np.floor)
df['is_latest_export'] = df['Date'] == latest_export_date
df_max_date = df[df['Date'] == latest_export_date]
df_max_date['in_latest_export'] = True
df = pd.merge(df, df_max_date, how="left", on=["UID"],suffixes = [None,'_current'])
df=df.round(2)




C:\Users\jackj\AppData\Local\Temp\ipykernel_10300\301999807.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [23]:
df.head(14)

,UID,Name,Age,Position,DoB,Nat,Apps,Av Rat,Acc,Aer,Agg,Agi,Ant,Bal,Bra,Cmd,Com,Cmp,Cnt,Cor,Cro,Dec,Det,Dri,Ecc,Fin,Fir,Fla,Fre,Han,Hea,...,1v1_current,Pac_current,Pas_current,Pen_current,Pos_current,Pun_current,Ref_current,TRO_current,Sta_current,Str_current,Tck_current,Tea_current,Tec_current,Thr_current,Vis_current,Wor_current,Date_current,Positions_current,gk_rating_current,of_rating_current,rating_current,physical_current,first_rating_current,first_physical_current,rating_growth_current,physical_growth_current,age_at_export_current,is_latest_export_current,in_latest_export,start_of_season,start_of_season_age
0,r-2002077763,Alex Harris,16.70,"AM (RL), ST (C)",2007-10-18 00:00:01,ENG,-,-,5,2,8,6,7,13,4,3,2,8,5,4,8,12,3,7,3,6,9,6,5,3,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,16
1,r-2002077755,Alexander Dreien,16.50,"AM (LC), ST (C)",2008-01-02 00:00:01,AUT,-,-,12,3,10,11,7,7,6,3,2,10,4,2,5,6,9,10,2,6,4,8,1,1,5,...,1.0,14.0,5.0,6.0,4.0,1.0,3.0,3.0,9.0,4.0,1.0,6.0,6.0,3.0,7.0,4.0,2030-01-01 00:00:01,"[AML, AMC, ST]",5.03,10.59,10.59,36.0,8.97,29.0,1.62,7.0,21.0,True,True,2024-01-01,15
2,2002077298,David Díaz,19.47,"AM (L), ST (C)",2005-01-11 00:00:01,ESP,-,-,6,2,6,5,5,7,7,1,3,4,6,3,3,7,5,5,1,5,6,7,3,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,18
3,2000231345,Joel Odei-Larbi,17.93,"AM (L), ST (C)",2006-07-27 00:00:01,ENG,-,-,14,3,9,12,6,8,7,2,4,6,4,5,5,5,15,7,1,7,5,13,5,2,5,...,2.0,20.0,4.0,4.0,2.0,2.0,3.0,1.0,8.0,7.0,2.0,3.0,5.0,3.0,4.0,6.0,2030-01-01 00:00:01,"[AML, ST]",4.25,10.69,10.69,43.0,10.66,41.0,0.03,2.0,23.0,True,True,2024-01-01,17
4,43273316,Muhammadu Faal,27.00,"AM (RL), ST (C)",1997-07-01 00:00:01,GAM,-,-,14,3,8,14,5,12,7,2,2,9,3,4,8,6,13,11,2,12,9,14,3,2,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,26
5,r-2002077759,Fabrice Onana,16.38,"AM (R), ST (C)",2008-02-14 00:00:01,GER,-,-,14,2,6,11,3,5,9,2,2,4,4,3,6,2,10,6,3,6,9,13,4,3,3,...,2.0,17.0,9.0,4.0,2.0,2.0,1.0,2.0,7.0,1.0,1.0,13.0,12.0,1.0,4.0,9.0,2030-01-01 00:00:01,"[AMR, ST]",3.92,9.72,9.72,39.0,8.47,30.0,1.25,9.0,21.0,True,True,2024-01-01,15
6,r-2002077753,Christopher Effiong,16.06,AM (RLC),2008-06-09 00:00:01,ENG,-,-,13,2,11,9,10,6,5,3,1,6,6,4,7,9,10,6,1,7,7,11,2,2,4,...,2.0,17.0,6.0,3.0,2.0,1.0,1.0,4.0,6.0,2.0,3.0,2.0,6.0,2.0,6.0,3.0,2030-01-01 00:00:01,"[AMR, AML, AMC]",4.32,9.75,9.75,38.0,8.88,31.0,0.88,7.0,21.0,True,True,2024-01-01,15
7,2000148932,Billie Busari,23.24,"M (R), AM (RL)",2001-04-06 00:00:01,ENG,-,-,11,1,11,12,8,6,2,3,1,9,7,7,7,12,9,9,2,8,11,13,6,2,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,22
8,2002073684,Dion Pressman,18.48,M (L),2006-01-08 00:00:01,ENG,-,-,7,2,7,7,5,7,7,3,3,6,5,4,5,6,6,5,2,5,6,6,6,3,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,17
9,r-2002077756,Abraham Njoku,16.48,M (RC),2008-01-09 00:00:01,NGA,-,-,11,2,7,8,4,5,6,2,2,6,4,5,6,9,10,4,3,7,12,8,4,1,3,...,2.0,13.0,14.0,8.0,3.0,1.0,1.0,1.0,7.0,5.0,6.0,5.0,8.0,1.0,10.0,10.0,2030-01-01 00:00:01,"[MR, MC]",4.30,8.53,8.53,33.0,7.12,26.0,1.41,7.0,21.0,True,True,2024-01-01,15


In [24]:
import dash
from dash import dcc, html, Input, Output, dash_table
from jupyter_dash import JupyterDash
import pandas as pd
import ast



# Select only the columns you want to display
columns_to_display = [
    "Name", "start_of_season_age", "rating", "rating_growth", 
    "physical", "physical_growth", "Acc", "Pac", "Jum", "is_latest_export", "Positions"
]
df = df[columns_to_display]

# Convert 'Positions' column to a space-separated string before creating Dash table
df['Positions'] = df['Positions'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# Initialize Dash app inside Jupyter
app = JupyterDash(__name__)

app.layout = html.Div([
    html.H1("Player Data Dashboard", style={"text-align": "center"}),

    # Create a row to hold the table and sidebar
    html.Div([
        # Main content (the table)
        html.Div(id="filtered-table", style={"width": "80%", "display": "inline-block"}),

        # Sidebar on the right side
        html.Div([
            html.H4("Filters", style={"text-align": "center"}),

            # Dropdown for is_latest_export filter
            dcc.Dropdown(
                id='is-latest-export-filter',
                options=[
                    {"label": "True", "value": True},
                    {"label": "False", "value": False}
                ],
                value=[True],  # Default filter value
                multi=True,
                style={"width": "200px", "margin-top": "20px"}
            ),

            # RangeSlider for start_of_season_age filter
            html.Div([
                html.Label("Select Age Range"),
                dcc.RangeSlider(
                    id='age-range-slider',
                    min=df['start_of_season_age'].min(),
                    max=df['start_of_season_age'].max(),
                    step=1,
                    marks={i: str(i) for i in range(df['start_of_season_age'].min(), df['start_of_season_age'].max() + 1, 5)},  # Only show every 5th year
                    value=[df['start_of_season_age'].min(), df['start_of_season_age'].max()],
                    allowCross=False
                )
            ], style={"margin-top": "30px", "width": "250px"}),

            # Multi-select filter for Positions
            html.Div([
                html.Label("Select Positions"),
                dcc.Dropdown(
                    id='positions-filter',
                    options=[
                        {"label": "GK", "value": "GK"},
                        {"label": "DR", "value": "DR"},
                        {"label": "DL", "value": "DL"},
                        {"label": "DC", "value": "DC"},
                        {"label": "DM", "value": "DM"},
                        {"label": "WBR", "value": "WBR"},
                        {"label": "WBL", "value": "WBL"},
                        {"label": "MR", "value": "MR"},
                        {"label": "MC", "value": "MC"},
                        {"label": "ML", "value": "ML"},
                        {"label": "AMC", "value": "AMC"},
                        {"label": "AMR", "value": "AMR"},
                        {"label": "AML", "value": "AML"},
                        {"label": "ST", "value": "ST"},
                        {"label": "All", "value": "All"}
                    ],
                    value=["All"],  # Default filter value
                    multi=True,
                    style={"width": "200px", "margin-top": "20px"}
                )
            ], style={"margin-top": "30px", "width": "250px"}),

        ], style={"width": "20%", "display": "inline-block", "padding": "20px", "border-left": "2px solid #ccc", "height": "100vh"})
    ], style={"display": "flex", "justify-content": "space-between"}),

])

# Callback to update the table based on the filters
@app.callback(
    Output("filtered-table", "children"),
    [
        Input("is-latest-export-filter", "value"),
        Input("age-range-slider", "value"),
        Input("positions-filter", "value")
    ]
)
def update_table(is_latest_export_values, age_range, positions_selected):
    # Filter the DataFrame based on the dropdown and range slider selections
    filtered_df = df[df['is_latest_export'].isin(is_latest_export_values)]
    filtered_df = filtered_df[filtered_df['start_of_season_age'].between(age_range[0], age_range[1])]
    
    # Filter based on positions selection
    if "All" not in positions_selected:  # If "All" is not selected, filter based on selected positions
        # Update the logic to check for exact match with spaces
        filtered_df = filtered_df[filtered_df['Positions'].apply(
            lambda x: any(f" {pos} " in f" {x} " for pos in positions_selected))]  # Add spaces at the start and end of the string
    
    # Display the DataFrame in the DataTable and enable sorting
    return dash_table.DataTable(
        data=filtered_df.to_dict("records"),  # Data for the table
        columns=[{"name": i, "id": i} for i in filtered_df.columns],  # Column names
        style_table={'overflowX': 'auto'},
        sort_action="native",  # Enable sorting of columns
        sort_mode="multi",  # Allow multiple columns to be sorted at once
        style_cell={'textAlign': 'center'},  # Center-align the text in the table
    )



C:\Users\jackj\anaconda3\Lib\site-packages\dash\dash.py:585: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



In [25]:
if __name__ == '__main__':
    app.run(debug=True,jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
